In [ ]:
import cv2
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

BASE_DIR = os.path.abspath("..")

INPUT_DIR = os.path.join(BASE_DIR, "data/WLBisindo/raw")
OUTPUT_DIR = os.path.join(BASE_DIR, "data/WLBisindo/frames")
CSV_PATH = os.path.join(BASE_DIR, "data/WLBisindo/classes.csv")

TARGET_FRAMES = 20

In [ ]:
def load_label_map(csv_path):
    df = pd.read_csv(csv_path)
    return dict(zip(df["label"], df["gloss"]))

label_map = load_label_map(CSV_PATH)

print("Total kelas:", len(label_map))

In [ ]:
def get_label_from_filename(filename):
    parts = filename.split("_")
    for part in parts:
        if "label" in part:
            return int(part.replace("label", ""))
    return None

In [ ]:
def is_motion(frame1, frame2, threshold=50000):
    diff = cv2.absdiff(frame1, frame2)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    score = np.sum(gray)
    return score > threshold


def is_not_blurry(frame, threshold=100):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    var = cv2.Laplacian(gray, cv2.CV_64F).var()
    return var > threshold


def uniform_sample(frames, num_samples):
    if len(frames) == 0:
        return []

    if len(frames) <= num_samples:
        return frames

    idxs = np.linspace(0, len(frames) - 1, num_samples).astype(int)
    return [frames[i] for i in idxs]

In [ ]:
def process_video(video_path, output_folder):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames < 5:
        return

    start = int(total_frames * 0.3)
    end = int(total_frames * 0.7)

    frames = []
    prev_frame = None

    cap.set(cv2.CAP_PROP_POS_FRAMES, start)

    for i in range(start, end):
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (224, 224))

        if prev_frame is None:
            prev_frame = frame
            continue

        if not is_motion(prev_frame, frame):
            prev_frame = frame
            continue

        if not is_not_blurry(frame):
            prev_frame = frame
            continue

        frames.append(frame)
        prev_frame = frame

    cap.release()

    frames = uniform_sample(frames, TARGET_FRAMES)

    if len(frames) == 0:
        return

    os.makedirs(output_folder, exist_ok=True)

    for i, frame in enumerate(frames):
        cv2.imwrite(os.path.join(output_folder, f"frame_{i}.jpg"), frame)

In [ ]:
def process_dataset():
    files = os.listdir(INPUT_DIR)

    for file in tqdm(files):
        if not file.endswith(".mp4"):
            continue

        label_id = get_label_from_filename(file)
        if label_id is None:
            continue

        if label_id not in label_map:
            continue

        label_name = label_map[label_id]

        video_name = os.path.splitext(file)[0]

        output_folder = os.path.join(
            OUTPUT_DIR,
            label_name,
            video_name
        )

        video_path = os.path.join(INPUT_DIR, file)

        process_video(video_path, output_folder)

In [ ]:
process_dataset()